# Graph Laplacian and Spectral Graph Theory

The **graph Laplacian** $L = D - W$ encodes the topology of a weighted graph $G = (V, E, w)$:
- $W_{ij} = w_{ij}$ (edge weights, $W_{ij} = 0$ if $(i,j) \notin E$).
- $D = \mathrm{diag}(d_i)$ where $d_i = \sum_j W_{ij}$ (degree matrix).

## Properties

- $L$ is symmetric, positive semidefinite: $x^\top L x = \frac{1}{2}\sum_{ij} w_{ij}(x_i - x_j)^2 \ge 0$.
- $L \mathbf{1} = 0$ (constant functions are in the nullspace).
- Number of zero eigenvalues = number of connected components.
- Eigenvectors of $L$ are the **graph Fourier basis**; the $k$-th eigenvector is the $k$-th Fourier mode.

## Normalized Laplacian

The **normalized Laplacian** $\mathcal{L} = D^{-1/2} L D^{-1/2}$ has eigenvalues in $[0, 2]$ and is used in spectral clustering.

## Laplacian as covariance

For a Gaussian Markov random field, the precision (inverse covariance) matrix is $\Sigma^{-1} = L + \lambda I$ (regularised Laplacian). Thus $\Sigma = (L + \lambda I)^{-1}$ gives a covariance structure where nearby nodes are correlated.

## Spectral clustering

The **Fiedler vector** (eigenvector of the second-smallest eigenvalue $\lambda_2$) bipartitions the graph optimally in a relaxed sense. Clustering into $k$ groups uses the $k$ smallest eigenvectors (spectral embedding + $k$-means).

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags, csr_matrix
from scipy.sparse.linalg import eigsh
from sklearn.neighbors import kneighbors_graph
from ipywidgets import interact, IntSlider, FloatLogSlider

plt.rcParams['figure.dpi'] = 120

## k-NN graph and Laplacian

We build a $k$-nearest-neighbour graph on 2D data and compute the Laplacian spectrum.

In [ ]:
rng = np.random.default_rng(0)
# Two-moon dataset
n = 80
theta1 = rng.uniform(0, np.pi, n//2)
theta2 = rng.uniform(np.pi, 2*np.pi, n//2)
pts = np.vstack([
    np.column_stack([np.cos(theta1), np.sin(theta1)]),
    np.column_stack([np.cos(theta2) + 1, np.sin(theta2) - 0.5]),
])
pts += 0.05 * rng.standard_normal(pts.shape)

k_nn = 8
A = kneighbors_graph(pts, k_nn, mode='connectivity', include_self=False)
W = (A + A.T).toarray() / 2   # symmetrize
np.fill_diagonal(W, 0)
d = W.sum(axis=1)
D = np.diag(d)
L = D - W
print(f'Graph: {len(pts)} nodes, {int(W.sum()/2)} edges')

# Spectrum
eigenvalues, eigenvectors = np.linalg.eigh(L)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Graph
axes[0].scatter(pts[:,0], pts[:,1], c='steelblue', s=60, zorder=5)
for i in range(n):
    for j in range(i+1, n):
        if W[i,j] > 0:
            axes[0].plot(pts[[i,j],0], pts[[i,j],1], 'k-', lw=0.4, alpha=0.3)
axes[0].set_aspect('equal'); axes[0].axis('off'); axes[0].set_title('k-NN graph')

# Spectrum
axes[1].plot(eigenvalues[:20], 'o-', color='tomato', ms=6)
axes[1].set_xlabel('index'); axes[1].set_ylabel('eigenvalue $\\lambda_k$')
axes[1].set_title('Laplacian spectrum (first 20)'); axes[1].grid(alpha=0.3)

# Fiedler vector (2nd eigenvector)
fiedler = eigenvectors[:, 1]
col_map = plt.cm.RdBu(0.5 + fiedler / (2*np.abs(fiedler).max()))
for i in range(n):
    for j in range(i+1, n):
        if W[i,j] > 0:
            axes[2].plot(pts[[i,j],0], pts[[i,j],1], 'k-', lw=0.4, alpha=0.3)
sc = axes[2].scatter(pts[:,0], pts[:,1], c=fiedler, cmap='RdBu', s=80, zorder=5)
plt.colorbar(sc, ax=axes[2], fraction=0.046)
axes[2].set_aspect('equal'); axes[2].axis('off')
axes[2].set_title('Fiedler vector $v_2$ (spectral bisection)')

plt.tight_layout(); plt.show()

## Laplacian as inverse covariance

The regularised Laplacian $\Sigma^{-1} = L + \lambda I$ defines a Gaussian Markov random field. We draw samples and visualise the spatial correlation structure.

In [ ]:
lam = 0.5
Sigma = np.linalg.inv(L + lam * np.eye(n))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Covariance matrix
im = axes[0].imshow(Sigma, cmap='RdBu_r', aspect='auto')
plt.colorbar(im, ax=axes[0], fraction=0.046)
axes[0].set_title('Covariance $\\Sigma = (L+\\lambda I)^{-1}$')

# Draw samples
samples = rng.multivariate_normal(np.zeros(n), Sigma, 3)
for ax, s in zip(axes[1:], samples[:2]):
    for i in range(n):
        for j in range(i+1, n):
            if W[i,j] > 0:
                ax.plot(pts[[i,j],0], pts[[i,j],1], 'k-', lw=0.4, alpha=0.3)
    sc = ax.scatter(pts[:,0], pts[:,1], c=s, cmap='viridis', s=80, zorder=5)
    plt.colorbar(sc, ax=ax, fraction=0.046)
    ax.set_aspect('equal'); ax.axis('off'); ax.set_title('Sample from GMRF')

plt.tight_layout(); plt.show()

## Spectral clustering

Using the $k$ smallest eigenvectors of $\mathcal{L}$ as node features, then $k$-means yields the spectral clustering partition.

In [ ]:
from sklearn.cluster import KMeans

# Normalized Laplacian
d_inv_sqrt = np.diag(1.0 / np.sqrt(np.maximum(d, 1e-12)))
L_norm = d_inv_sqrt @ L @ d_inv_sqrt
evals_n, evecs_n = np.linalg.eigh(L_norm)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, k_clust in zip(axes, [2, 3, 4]):
    U = evecs_n[:, :k_clust]
    U_norm = U / (np.linalg.norm(U, axis=1, keepdims=True) + 1e-12)
    labels = KMeans(n_clusters=k_clust, n_init=10, random_state=0).fit_predict(U_norm)
    cols_k = plt.cm.tab10(labels / 9.0)
    for i in range(n):
        for j in range(i+1, n):
            if W[i,j] > 0:
                ax.plot(pts[[i,j],0], pts[[i,j],1], 'k-', lw=0.4, alpha=0.2)
    ax.scatter(pts[:,0], pts[:,1], c=cols_k, s=80, edgecolors='k', linewidths=0.5, zorder=5)
    ax.set_aspect('equal'); ax.axis('off'); ax.set_title(f'Spectral clustering $k={k_clust}$')

plt.tight_layout(); plt.show()

## Interactive: regularisation $\lambda$

In [ ]:
def show_gmrf(log_lam=0.0, k_nn2=8):
    A2 = kneighbors_graph(pts, k_nn2, mode='connectivity', include_self=False)
    W2 = (A2 + A2.T).toarray() / 2; np.fill_diagonal(W2, 0)
    D2 = np.diag(W2.sum(axis=1))
    L2 = D2 - W2
    lam2 = 10**log_lam
    Sig2 = np.linalg.inv(L2 + lam2*np.eye(n))
    s = rng.multivariate_normal(np.zeros(n), Sig2)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    im = axes[0].imshow(Sig2, cmap='RdBu_r', aspect='auto')
    plt.colorbar(im, ax=axes[0], fraction=0.046)
    axes[0].set_title(f'Covariance ($\\lambda={lam2:.3f}$, k={k_nn2})')
    for i in range(n):
        for j in range(i+1,n):
            if W2[i,j] > 0:
                axes[1].plot(pts[[i,j],0], pts[[i,j],1],'k-',lw=0.4,alpha=0.3)
    sc = axes[1].scatter(pts[:,0], pts[:,1], c=s, cmap='viridis', s=80, zorder=5)
    plt.colorbar(sc, ax=axes[1], fraction=0.046)
    axes[1].set_aspect('equal'); axes[1].axis('off'); axes[1].set_title('GMRF sample')
    plt.tight_layout(); plt.show()

interact(show_gmrf,
         log_lam=FloatLogSlider(value=0.0, min=-2.0, max=2.0, step=0.25, description='$\\log_{10}\\lambda$'),
         k_nn2=IntSlider(value=8, min=3, max=20, step=1, description='$k$-NN'));

## Bibliographical resources

- Chung, F. R. K. (1997). *Spectral Graph Theory*. American Mathematical Society.
- von Luxburg, U. (2007). A tutorial on spectral clustering. *Statistics and Computing*, 17(4), 395–416.
- Ng, A. Y., Jordan, M. I. and Weiss, Y. (2002). On spectral clustering: Analysis and an algorithm. *Advances in Neural Information Processing Systems*, 14.
- Belkin, M. and Niyogi, P. (2003). Laplacian eigenmaps for dimensionality reduction. *Neural Computation*, 15(6), 1373–1396.
- Mohar, B. (1991). The Laplacian spectrum of graphs. In *Graph Theory, Combinatorics, and Applications* (pp. 871–898). Wiley.